In [0]:
USE CATALOG workspace;
USE SCHEMA ecom_poc;

In [0]:
CREATE OR REPLACE TABLE bronze_orders
COMMENT 'Raw orders, all columns as STRING, loaded unchanged from orders.csv'
AS
SELECT
  order_id, customer_id, order_date, category,
  quantity, unit_price, order_status, country,
  current_timestamp()          AS _ingested_at,
  _metadata.file_path          AS _source_file,
  'orders_batch_20261118_001'  AS _batch_id,
  sha2(concat_ws('|', order_id, customer_id, order_date, category,
                 quantity, unit_price, order_status, country), 256) AS _row_hash
FROM read_files(
  '/Volumes/workspace/ecom_poc/raw_files/orders.csv',
  format    => 'csv',
  header    => true,
  schema    => 'order_id STRING, customer_id STRING, order_date STRING, category STRING, quantity STRING, unit_price STRING, order_status STRING, country STRING',
  nullValue => '__NO_NULL__'
);

In [0]:
CREATE OR REPLACE TABLE bronze_transactions
COMMENT 'Raw transactions, all columns as STRING, loaded unchanged from transactions.csv'
AS
SELECT
  txn_id, order_id, payment_method, amount, currency, txn_status, txn_ts,
  current_timestamp()                AS _ingested_at,
  _metadata.file_path                AS _source_file,
  'transactions_batch_20261118_001'  AS _batch_id,
  sha2(concat_ws('|', txn_id, order_id, payment_method, amount,
                 currency, txn_status, txn_ts), 256) AS _row_hash
FROM read_files(
  '/Volumes/workspace/ecom_poc/raw_files/transactions.csv',
  format    => 'csv',
  header    => true,
  schema    => 'txn_id STRING, order_id STRING, payment_method STRING, amount STRING, currency STRING, txn_status STRING, txn_ts STRING',
  nullValue => '__NO_NULL__'
);

In [0]:
SELECT 'orders' AS dataset,
       (SELECT COUNT(*) FROM read_files('/Volumes/workspace/ecom_poc/raw_files/orders.csv',
            format => 'csv', header => true)) AS source_rows,
       (SELECT COUNT(*) FROM bronze_orders) AS bronze_rows
UNION ALL
SELECT 'transactions',
       (SELECT COUNT(*) FROM read_files('/Volumes/workspace/ecom_poc/raw_files/transactions.csv',
            format => 'csv', header => true)),
       (SELECT COUNT(*) FROM bronze_transactions);

In [0]:
DESCRIBE TABLE bronze_orders;
SELECT * FROM bronze_orders LIMIT 10;
DESCRIBE TABLE bronze_transactions;
SELECT * FROM bronze_transactions LIMIT 10;